# Анализ настройки коэффициентов PID (подбор значений нейросетью)

Объединяет анализ ранних TD3-экспериментов, где агент напрямую подбирает значения коэффициентов PID. 

Эксперименты:
* `2025-08-28_15-38-42_td3_train_real_async` - свободный запуск из-за ошибки микроконтроллера
* `2025-08-29_19-46-18_td3_train_real_async` 
* `2025-09-09_13-17-51_td3_train_real_async` - другая архитектура нейронных сетей
* `2025-09-15_17-45-50_absolute_skip50` - абсолютная награда + усреднение по 50 шагам (⚠️ логи среды записаны неправильно)
* `2025-09-15_18-20-38_relative_skip1` - относительная награда (⚠️ логи среды записаны неправильно)
* `2025-09-15_19-04-16_relative_skip50` - относительная награда + усреднение по 50 шагам (⚠️ логи среды записаны неправильно)
* `2025-09-24_16-43-48_relative_skip1` - то же + система повышения минимального напряжения (повторный прогон)
* `2025-09-24_18-14-49_absolute_skip50` - то же + система повышения минимального напряжения
* `2025-09-24_18-31-57_relative_skip1` - то же + система повышения минимального напряжения
* `2025-09-24_18-52-01_relative_skip50` - то же + система повышения минимального напряжения

In [ ]:
%load_ext autoreload
%autoreload 2

## Описание эксперимента

### Алгоритм

Использовался **TD3** (Twin Delayed DDPG), адаптированный для работы в реальном времени в среде с бесконечным горизонтом (`done` всегда `False`). Особенности: два Q-критика (при оценке цели берётся минимум), задержанное обновление актора (`update_actor_freq`), мягкое обновление target-сетей (`SoftUpdate`), сглаживание целевой политики (к действию target-актора добавляется обрезанный гауссов шум `policy_noise=0.2`, `noise_clip=0.5`). Сбор данных и обучение разделены (асинхронный коллектор).

**Шум исследования не применялся:** в коллектор передавался детерминированный актор (`AdditiveGaussianModule` импортируется, но не вызывается; параметров `exploration_sigma_*` в конфигах нет). Шум присутствовал только как сглаживание целевой политики внутри `TD3Loss` и к реальным действиям на установке не подмешивался. Разнообразие данных обеспечивалось случайной инициализацией сети и её недообученностью на старте.

### Архитектура сетей

- **Actor:** `[LSTM →] MLP → TanhModule`. На `28.08.25` и `29.08.25` — LSTM + MLP (`use_lstm=True`); с `09.09.25` и далее — только MLP (ReLU, `mlp_num_cells`, напр. `[256, 256]`).
- **Critic (Q-функция):** MLP, вход `[action, observation]`, выход 1. Два критика (`num_qvalue_nets=2`). LSTM в критике опционален и в этих прогонах выключен.

### Окружение

`PidTuningExperimentalEnv` — обмен с установкой по COM (`→ kp ki kd [u_min u_max]`, `← PV CO`).

**Состояние** $s$ (3 компоненты): $[\,\text{PV}_\text{norm},\ \text{CO}_\text{norm},\ \text{setpoint}_\text{norm}\,]$ — сигнал АЦП (process variable), сигнал ЦАП (control output), уставка. Нормировка АЦП: $U/10230\cdot 2 - 1$; ЦАП: $U/4095\cdot 2 - 1$, обе в $[-1, 1]$.

**Действие** $a$ (3 компоненты): $[K_p, K_i, K_d]$. Финальный `TanhModule(spec=action_spec)` отображает выход сети $z$ в физический диапазон: $\tanh(z)\in[-1,1]$, затем аффинно в $[\text{low}, \text{high}]$ из `env.bounds.action`:
$$ a = \text{low} + (\text{high} - \text{low})\cdot\frac{\tanh(z) + 1}{2}. $$
Для серий 15.09/24.09: $K_p\in[0,\ 17.5]$, $K_i\in[0,\ 55]$, $K_d\in[0,\ 0.01]$. В прошивку идут уже физические коэффициенты.

**Награда** $r$ могла рассчитываться по одной из функций:

* **Абсолютная ошибка (по умолчанию)** — на нормированных значениях:
$$ r = -\left\lvert U_\text{setpoint} - U_\text{process variable} \right\rvert + 1, $$
награда в диапазоне $[-1, 1]$: чем меньше ошибка, тем ближе к $1$.

* **Относительная ошибка** — на сырых значениях АЦП:
$$ \text{relative\_error} = \frac{\left\lvert U_\text{setpoint} - U_\text{process variable} \right\rvert}{\left\lvert U_\text{setpoint} \right\rvert + \varepsilon}, \quad r = \max\{1 - \text{relative\_error},\ -1\}, $$
где $\varepsilon$ — малый коэффициент для защиты от деления на ноль; жёстче штрафует ошибки.

* **Экспоненциальная ошибка** — на нормированных значениях:
$$ r = 2\exp\big(-k \cdot \lvert U_\text{setpoint} - U_\text{process variable} \rvert\big) - 1, $$
где $k > 0$ задаёт жёсткость штрафа.

Для `absolute`/`exponential` ошибка берётся по нормированным значениям, для `relative` — по сырым (из-за сдвига $-1$ в нормировке относительная ошибка на нормированных значениях не совпала бы с обучением).

**Важно:** значение награды в логи не записывалось, поэтому при анализе оно восстанавливается по формуле, соответствующей типу эксперимента (см. `reward_type` из конфига). Точное значение, использованное при обучении (с усреднением по N шагов), воспроизвести нельзя.

### Трансформации сбора данных

- **`StepsAggregateTransform(frame_skip=N)`** — одно и то же действие применяется к установке $N$ раз; наблюдение и награда **усредняются** (mean) по этим $N$ «сырым» шагам, агент видит один переход. Поэтому в env-логах `x`/`setpoint`/`reward` уже усреднены за $N$ шагов (при `skip50` — за 50), а не мгновенные.
- **`InitialActionRepeatTransform(repeat_count=M)`** — одноразовый прогрев в начале: первое действие применяется $M$ раз, промежуточные шаги отбрасываются (система выходит на режим под стартовыми коэффициентами). Срабатывает один раз за прогон (горизонт бесконечный, сбросов нет).

## Загрузка данных

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from enum import Enum
from pathlib import Path
import re

import numpy as np
import pandas as pd
from omegaconf import DictConfig, OmegaConf

from nn_laser_stabilizer.utils.paths import get_experiment_dir
from nn_laser_stabilizer.analysis.sources import read_tfevents, parse_keyval_log

# Сигналы в логах нормированы в [-1, 1]; константы для перевода в физ. единицы.
ADC_MAX = 10230  # process variable / setpoint (АЦП)
DAC_MAX = 4095   # control output (ЦАП)

REWARD_EPS = 1e-6           # защита от деления на ноль в relative (как в коде среды)
REWARD_EXP_K_DEFAULT = 1.0  # k для exponential, если не задан в конфиге


class RewardType(str, Enum):
    ABSOLUTE = "absolute"
    RELATIVE = "relative"
    EXPONENTIAL = "exponential"


@dataclass
class ExperimentData:
    env_df: pd.DataFrame
    train_df: pd.DataFrame
    duration_sec: float
    reward_type: RewardType
    config: DictConfig


def _reward_from_config(config: DictConfig) -> tuple[RewardType, float]:
    """Тип награды и параметр k из конфига. В ранних конфигах env.reward нет → absolute."""
    name = OmegaConf.select(config, "env.reward.name", default=None)
    if name is None:
        return RewardType.ABSOLUTE, REWARD_EXP_K_DEFAULT
    k = OmegaConf.select(config, "env.reward.args.k", default=REWARD_EXP_K_DEFAULT)
    return RewardType(name), float(k)


def _denormalize_adc(v: pd.Series) -> pd.Series:
    """Обратное преобразование нормировки АЦП: [-1, 1] -> сырое значение."""
    return (v + 1.0) / 2.0 * ADC_MAX


def _compute_reward(
    setpoint: pd.Series, x: pd.Series, reward_type: RewardType, k: float
) -> pd.Series:
    """Восстановление награды (reward не логируется).

    absolute / exponential считаются на нормированных значениях, relative — на
    сырых (АЦП), как в коде среды: из-за сдвига −1 в нормировке относительная
    ошибка на нормированных значениях не совпала бы с обучением.
    """
    error = (setpoint - x).abs()
    if reward_type is RewardType.ABSOLUTE:
        return -error + 1
    if reward_type is RewardType.RELATIVE:
        setpoint_raw = _denormalize_adc(setpoint)
        x_raw = _denormalize_adc(x)
        relative_error = (setpoint_raw - x_raw).abs() / (setpoint_raw.abs() + REWARD_EPS)
        return (1 - relative_error).clip(lower=-1)
    if reward_type is RewardType.EXPONENTIAL:
        return 2 * np.exp(-k * error) - 1
    raise ValueError(f"Неизвестный тип награды: {reward_type}")


def _finalize_env(
    env_df: pd.DataFrame, reward_type: RewardType, reward_k: float
) -> pd.DataFrame:
    """reward по нормированным значениям, затем перевод сигналов в физ. единицы."""
    env_df = env_df.copy()
    env_df["reward"] = _compute_reward(
        env_df["Observation/setpoint"], env_df["Observation/x"], reward_type, reward_k
    )

    to_adc = lambda v: ((v + 1.0) / 2) * ADC_MAX
    env_df["Observation/x"] = env_df["Observation/x"].apply(to_adc)
    env_df["Observation/setpoint"] = env_df["Observation/setpoint"].apply(to_adc)
    if "Observation/control_output" in env_df.columns:
        env_df["Observation/control_output"] = env_df[
            "Observation/control_output"
        ].apply(lambda v: ((v + 1.0) / 2) * DAC_MAX)
    return env_df


def _load_env_tfevents(exp_dir: Path) -> pd.DataFrame:
    return read_tfevents(str(exp_dir / "env_logs"))


def _load_env_keyval(exp_dir: Path) -> pd.DataFrame:
    # env-лог: step [time] kp ki kd x control_output setpoint -> исторические имена.
    # Колонка time есть не во всех поколениях — берём её, если присутствует.
    rename = {
        "step": "step",
        "kp": "Action/kp",
        "ki": "Action/ki",
        "kd": "Action/kd",
        "x": "Observation/x",
        "control_output": "Observation/control_output",
        "setpoint": "Observation/setpoint",
    }
    raw = parse_keyval_log(exp_dir / "env_logs" / "log.txt")
    if "time" in raw.columns:
        rename = {"step": "step", "time": "time",
                  **{k: v for k, v in rename.items() if k != "step"}}
    return (
        raw.rename(columns=rename)[list(rename.values())]
        .dropna()
        .reset_index(drop=True)
    )


_LOG_TS_RE = re.compile(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}),(\d{3})\]")


def _parse_log_time(line: str):
    m = _LOG_TS_RE.match(line)
    if not m:
        return None
    return datetime.strptime(m.group(1), "%Y-%m-%d %H:%M:%S").replace(
        microsecond=int(m.group(2)) * 1000
    )


def _duration_from_main_log(exp_dir: Path) -> float:
    """Длительность эксперимента: 'Training started' -> 'Training finished'."""
    log_path = next(exp_dir.glob("*.log"))
    started = finished = None
    with open(log_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            if "Training started" in line:
                started = _parse_log_time(line)
            elif "Training finished" in line:
                finished = _parse_log_time(line)

    if started is None or finished is None:
        raise ValueError(
            f"В логе {log_path} не найдены метки "
            "'Training started' и/или 'Training finished'"
        )
    return (finished - started).total_seconds()


def load_experiment(
    experiment_name: str, experiment_date: str, experiment_time: str
) -> ExperimentData:
    exp_dir = get_experiment_dir(
        experiment_name=experiment_name,
        experiment_date=experiment_date,
        experiment_time=experiment_time,
    )

    config = OmegaConf.load(exp_dir / ".hydra" / "config.yaml")
    reward_type, reward_k = _reward_from_config(config)

    # switch только по формату env-логов: tfevents у старых запусков, keyval у остальных.
    match (experiment_name, experiment_date, experiment_time):
        case ("td3_train_real_async", "2025-08-28", "15-38-42") | ("td3_train_real_async", "2025-08-29", "19-46-18"):
            env_df = _load_env_tfevents(exp_dir)
        case _:
            env_df = _load_env_keyval(exp_dir)

    env_df = _finalize_env(env_df, reward_type, reward_k)
    train_df = read_tfevents(str(exp_dir / "train_logs"))
    duration_sec = _duration_from_main_log(exp_dir)
    return ExperimentData(
        env_df=env_df,
        train_df=train_df,
        duration_sec=duration_sec,
        reward_type=reward_type,
        config=config,
    )

In [ ]:
data = load_experiment(
    experiment_name="td3_train_real_async",
    experiment_date="2025-08-28",
    experiment_time="15-38-42",
)
env_df = data.env_df
train_df = data.train_df

print(f"Длительность эксперимента: {data.duration_sec / 60:.1f} мин")
print(f"Тип награды: {data.reward_type.value}")
print(f"env_df: {len(env_df)} строк; колонки: {list(env_df.columns)}")
print(f"train_df: {len(train_df)} строк")

In [ ]:
# Полный конфиг эксперимента (.hydra/config.yaml).
print(OmegaConf.to_yaml(data.config))

In [ ]:
import matplotlib.pyplot as plt
from functools import wraps

# Все графики сохраняются в output/ по имени файла (путь в savefig игнорируется).
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

if not getattr(plt.savefig, "_auto_wrapped", False):
    _original_savefig = plt.savefig

    @wraps(_original_savefig)
    def _auto_savefig(filename, *args, **kwargs):
        filename = OUTPUT_DIR / Path(filename).name
        return _original_savefig(filename, *args, **kwargs)

    setattr(_auto_savefig, "_auto_wrapped", True)
    plt.savefig = _auto_savefig

## Анализ окружения

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for tag in ["Observation/x", "Observation/setpoint", "Observation/control_output"]:
    if tag in env_df.columns:
        ax.plot(env_df["step"], env_df[tag], label=tag, linewidth=0.8)
ax.set_title("Observation curves", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("Value")
ax.legend(title="Metric")
plt.tight_layout()
plt.savefig("observation_logs.pdf")
plt.show()

In [ ]:
obs_tags = [
    t for t in ["Observation/x", "Observation/setpoint", "Observation/control_output"]
    if t in env_df.columns
]
fig, axes = plt.subplots(len(obs_tags), 1, figsize=(10, 4 * len(obs_tags)), sharex=True)
axes = np.atleast_1d(axes)
for ax, tag in zip(axes, obs_tags):
    ax.plot(env_df["step"], env_df[tag], linewidth=0.8)
    ax.set_title(tag, fontsize=14)
    ax.set_ylabel("Value")
axes[-1].set_xlabel("Step")
plt.tight_layout()
plt.savefig("observation_logs_split.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(env_df["step"], env_df["reward"], label="reward", linewidth=0.8)
ax.set_title("Reward curve", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("Value")
ax.legend(title="Metric")
plt.tight_layout()
plt.savefig("reward_logs.pdf")
plt.show()

In [ ]:
tag = "Action/kp"
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(env_df["step"], env_df[tag], label=tag, linewidth=0.8)
ax.set_title(tag, fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("Value")
ax.legend(title="Metric")
plt.tight_layout()
plt.savefig("Action_kp.pdf")
plt.show()

In [ ]:
tag = "Action/ki"
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(env_df["step"], env_df[tag], label=tag, linewidth=0.8)
ax.set_title(tag, fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("Value")
ax.legend(title="Metric")
plt.tight_layout()
plt.savefig("Action_ki.pdf")
plt.show()

In [ ]:
tag = "Action/kd"
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(env_df["step"], env_df[tag], label=tag, linewidth=0.8)
ax.set_title(tag, fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("Value")
ax.legend(title="Metric")
plt.tight_layout()
plt.savefig("Action_kd.pdf")
plt.show()

In [ ]:
if "time" in env_df.columns:
    delta_ms = env_df["time"].diff() * 1000
    print("Интервал между шагами (мс):")
    print(delta_ms.describe())
else:
    print("В env-логе нет колонки time — статистика интервала недоступна.")

## Анализ обучения

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for tag in ["Loss/Critic", "Loss/Actor"]:
    if tag in train_df.columns:
        ax.plot(train_df["step"], train_df[tag], label=tag)
ax.set_title("Training curves", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("Value")
ax.legend(title="Metric")
plt.tight_layout()
plt.savefig("train_logs.pdf")
plt.show()

## Графики для докладов

Презентационные графики с захардкоженным оформлением под конкретные доклады (ВКВО, предзащита, статья/диплом). Время по оси рассчитывается из `data.duration_sec`. Рассматривается эксперимент `2025-08-28_15-38-42_td3_train_real_async`.

In [ ]:
step_max = env_df["step"].max()
env_df["time_sec"] = env_df["step"] / step_max * data.duration_sec
env_df["time_min"] = env_df["time_sec"] / 60

График для ВКВО.

In [ ]:
plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

SETPOINT = 1200

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(env_df["time_sec"], env_df["Observation/x"], color="green", label=r"$U_{\text{АЦП}}$")
ax1.axhline(SETPOINT, color="black", linestyle="--")
ax1.set_xlabel("Время (с)")
ax1.set_ylabel("Напряжение на фотодетекторе", color="green")
ax1.tick_params(axis="y", labelcolor="green")

plt.tight_layout()

График для отчёта по ВКР.

In [ ]:
plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

SETPOINT = 1200

fig, ax1 = plt.subplots(figsize=(10, 6))

u_mean = env_df["Observation/x"].mean()
u_std = env_df["Observation/x"].std()
print(f"U mean = {u_mean}; U std = {u_std}")

ax1.plot(env_df["time_sec"], env_df["Observation/x"], color="green", label=r"$U_{\text{АЦП}}$")
ax1.axhline(SETPOINT, color="black", linestyle="--")
ax1.set_xlabel("Время (с)")
ax1.set_ylabel("Напряжение на фотодетекторе (АЦП)", color="green")
ax1.tick_params(axis="y", labelcolor="green")
ax1.set_ylim(0, 2000)
ax1.set_xlim(1, 4000)
ax1.grid(True, alpha=0.3)

plt.tight_layout()

График для предзащиты.

In [ ]:
plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

plt.figure(figsize=(14, 7))

plt.plot(
    env_df["time_min"],
    env_df["Observation/x"],
    color="blue",
    alpha=0.7,
    linewidth=1,
)

plt.axhline(1200, linestyle="--", linewidth=5, color="black")
plt.ylim(600, 1800)

plt.xlabel("Время, мин")
plt.ylabel("Напряжение на фотодетекторе")

plt.grid(False)
plt.tight_layout()

plt.savefig("process_variable.pdf")
plt.savefig("process_variable.svg")
plt.savefig("process_variable.png")

plt.show()

График $K_d$ для ВКВО.

In [ ]:
DEFAULT_KD = 0.002

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(env_df["time_sec"], env_df["Action/kd"], color="blue", label=r"Предсказанное моделью значение $K_{\text{d}}$")
ax.axhline(DEFAULT_KD, color="black", linestyle="--", label=r"Текущее значение $K_{\text{d}}$")
ax.set_xlabel("Время (с)")
ax.set_ylabel(r"Коэффициент $K_{\text{d}}$", color="blue")
ax.tick_params(axis="y", labelcolor="blue")
ax.set_xlim(0, 300)

lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, loc="best")

plt.tight_layout()
plt.savefig("kd.png")

График для статьи и диплома.

In [ ]:
def hsl_to_rgb_normalized(h, s, l):
    # colorsys использует HLS (не HSL), поэтому порядок: (h, l, s)
    from colorsys import hls_to_rgb
    return hls_to_rgb(h / 360, l / 100, s / 100)

BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
GREEN_RGB = hsl_to_rgb_normalized(120, 72, 44)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)
PURPLE_RGB = hsl_to_rgb_normalized(279, 98, 76)
LIGHT_PURPLE_RGB = hsl_to_rgb_normalized(278, 100, 94)

GRAY_RGB = (123 / 255, 123 / 255, 123 / 255)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    env_df["time_min"],
    env_df["Observation/x"] / 10,
    color=BLUE_RGB,
    alpha=0.7,
    linewidth=1,
)

ax.axhline(120, linestyle="--", linewidth=5, label="Уставка", color=RED_RGB)

ax.set_xlabel("Время, мин")
ax.set_ylabel("Напряжение на фотодетекторе")
ax.set_xlim(left=0, right=42)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)

# Невидимая ось для выравнивания с графиком со стабилизацией.
ax2 = ax.twinx()
ax2.set_ylim(0, 3000)
ax2.set_ylabel("Напряжение на фазовращателе", color="white")
ax2.tick_params(right=True, labelright=True, colors="white")

plt.grid(False)
plt.tight_layout()

plt.savefig("signal_without_stab.pdf")
plt.show()